# Week 7: Tree Models & Ensemble Methods (RF, GBDT)
# 第 7 週：樹模型與集成方法（隨機森林、梯度提升）

## 學習目標 Learning Objectives
- 理解決策樹的建構原理與分裂準則 (Gini / Entropy)
- 視覺化決策樹結構與決策邊界
- 觀察 `max_depth` 對決策邊界與過擬合的影響
- 掌握隨機森林 (Random Forest) 的 OOB 分數與 n_estimators 的關係
- 理解梯度提升 (GBDT) 的學習率與迭代次數的交互影響
- 比較 Decision Tree / Random Forest / GBDT 的效能與決策邊界
- 視覺化特徵重要度 (Feature Importance)

In [ ]:
# 環境準備 Environment Setup
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import load_iris, load_wine, make_moons
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# 設定中文字型與繪圖風格
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)
print('所有套件載入成功！All packages imported successfully!')

---
## Part 1: Decision Tree Basics — 決策樹基礎

決策樹透過一系列 if-else 規則，將特徵空間遞迴地切割為越來越純的子區域。

A Decision Tree recursively splits the feature space into purer sub-regions using if-else rules.

**CART 分類樹**使用**基尼不純度 (Gini Impurity)** 作為預設分裂準則：

$$\text{Gini}(S) = 1 - \sum_{i=1}^{c} p_i^2$$

其中 $p_i$ 是類別 $i$ 在節點中的比例。Gini = 0 表示完全純淨，Gini = 0.5（二元分類）表示最不純。

In [ ]:
# Part 1: 在 Iris 資料集上建構並視覺化決策樹
iris = load_iris()
X_iris = iris.data[:, 2:4]  # 取花瓣長度和花瓣寬度（便於 2D 視覺化）
y_iris = iris.target
feature_names = ['Petal Length', 'Petal Width']
class_names = iris.target_names

# 訓練一棵決策樹
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_iris, y_iris)

# 以文字方式顯示樹結構
tree_text = export_text(dt, feature_names=feature_names)
print('決策樹結構 Decision Tree Structure:')
print(tree_text)
print(f'\n訓練準確率 Training Accuracy: {dt.score(X_iris, y_iris):.4f}')
print(f'樹深度 Tree Depth: {dt.get_depth()}')
print(f'葉節點數 Number of Leaves: {dt.get_n_leaves()}')

In [ ]:
# 視覺化決策邊界 Visualize Decision Boundary
def plot_decision_boundary(clf, X, y, ax=None, title='', class_names=None, alpha=0.3):
    """繪製分類器的決策邊界"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
    cmap_bold = ['#FF0000', '#00AA00', '#0000FF']

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=alpha)
    ax.contour(xx, yy, Z, colors='k', linewidths=0.5, alpha=0.5)

    markers = ['o', 's', '^']
    for idx in np.unique(y):
        mask = y == idx
        label = class_names[idx] if class_names is not None else f'Class {idx}'
        ax.scatter(X[mask, 0], X[mask, 1], c=cmap_bold[idx], marker=markers[idx],
                   edgecolors='k', s=40, label=label, zorder=3)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=8)
    return ax

fig, ax = plt.subplots(figsize=(9, 7))
plot_decision_boundary(dt, X_iris, y_iris, ax=ax,
                       title='決策樹決策邊界 (max_depth=3)\nDecision Tree Boundary (Iris)',
                       class_names=list(class_names))
ax.set_xlabel('Petal Length (cm)')
ax.set_ylabel('Petal Width (cm)')
plt.tight_layout()
plt.show()

print('觀察：決策樹的邊界是軸對齊的矩形區域 (axis-aligned rectangular regions)')

---
## Part 2: max_depth Comparison — 樹深度對決策邊界的影響

樹的深度 `max_depth` 控制模型的複雜度：
- 深度太淺 → 欠擬合 (Underfitting)
- 深度太深 → 過擬合 (Overfitting)

讓我們比較 depth = 1, 3, 5, None（不限制）四種情況。

In [ ]:
# Part 2: 比較不同 max_depth 的決策邊界
depths = [1, 3, 5, None]
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, depth in zip(axes, depths):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_iris, y_iris)
    acc = clf.score(X_iris, y_iris)
    n_leaves = clf.get_n_leaves()
    depth_val = clf.get_depth()

    depth_label = str(depth) if depth is not None else 'None'
    plot_decision_boundary(clf, X_iris, y_iris, ax=ax,
                           title=f'max_depth={depth_label}\nAcc={acc:.3f}, Leaves={n_leaves}, Depth={depth_val}',
                           class_names=list(class_names))
    ax.set_xlabel('Petal Length')
    ax.set_ylabel('Petal Width')

fig.suptitle('max_depth 對決策邊界的影響 — Effect of max_depth on Decision Boundary',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print('觀察重點：')
print('- depth=1:    只做一次分裂 (Decision Stump)，邊界非常簡單')
print('- depth=3:    中等複雜度，邊界開始捕捉資料結構')
print('- depth=5:    邊界更精細')
print('- depth=None: 完全生長，可能出現過擬合的小區域')

---
## Part 3: Gini vs Entropy — 分裂準則比較

兩種常見的分裂準則：

**基尼不純度 (Gini Impurity):**
$$\text{Gini}(S) = 1 - \sum_{i=1}^{c} p_i^2$$

**熵 (Entropy):**
$$H(S) = -\sum_{i=1}^{c} p_i \log_2 p_i$$

兩者在二元分類時的行為非常相似，但 Gini 計算更快（不需要對數運算）。

In [ ]:
# Part 3: 視覺化 Gini 與 Entropy 不純度函數
p = np.linspace(0.001, 0.999, 500)

# 二元分類的不純度計算
gini = 2 * p * (1 - p)
entropy = -p * np.log2(p) - (1 - p) * np.log2(1 - p)
misclass = 1 - np.maximum(p, 1 - p)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# 左圖：不純度函數比較
axes[0].plot(p, gini, 'b-', linewidth=2.5, label='Gini Impurity (2p(1-p))')
axes[0].plot(p, entropy, 'r-', linewidth=2.5, label='Entropy ($-p\\log_2 p - (1-p)\\log_2(1-p)$)')
axes[0].plot(p, misclass, 'g--', linewidth=2, label='Misclassification Error')
axes[0].set_xlabel('$p$ (proportion of class 1)', fontsize=12)
axes[0].set_ylabel('Impurity', fontsize=12)
axes[0].set_title('不純度函數比較\nImpurity Functions Comparison', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=0.5, color='gray', linestyle=':', alpha=0.5)
axes[0].text(0.52, 0.45, 'max impurity\nat p=0.5', fontsize=9, color='gray')

# 右圖：Gini vs Entropy 決策邊界
criteria = ['gini', 'entropy']
colors_cr = ['#2563eb', '#dc2626']
for i, (criterion, color) in enumerate(zip(criteria, colors_cr)):
    clf = DecisionTreeClassifier(criterion=criterion, max_depth=4, random_state=42)
    clf.fit(X_iris, y_iris)
    acc = clf.score(X_iris, y_iris)
    n_leaves = clf.get_n_leaves()
    depth_actual = clf.get_depth()
    x_pos = [n_leaves, depth_actual]
    labels_bar = ['Leaves', 'Depth']
    offset = 0.2 if i == 0 else -0.2
    bars = axes[1].bar([j + offset for j in range(2)], x_pos, width=0.35,
                       color=color, alpha=0.7, edgecolor='black',
                       label=f'{criterion.capitalize()} (Acc={acc:.3f})')
    for bar, val in zip(bars, x_pos):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                     str(val), ha='center', fontsize=10, fontweight='bold')

axes[1].set_xticks(range(2))
axes[1].set_xticklabels(['Number of Leaves', 'Tree Depth'], fontsize=11)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Gini vs Entropy 樹結構比較\nTree Structure Comparison', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Gini vs Entropy 決策邊界並排比較
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, criterion in zip(axes, ['gini', 'entropy']):
    clf = DecisionTreeClassifier(criterion=criterion, max_depth=4, random_state=42)
    clf.fit(X_iris, y_iris)
    plot_decision_boundary(clf, X_iris, y_iris, ax=ax,
                           title=f'criterion={criterion}\nAcc={clf.score(X_iris, y_iris):.3f}, '
                                 f'Leaves={clf.get_n_leaves()}',
                           class_names=list(class_names))
    ax.set_xlabel('Petal Length')
    ax.set_ylabel('Petal Width')

fig.suptitle('Gini vs Entropy 決策邊界比較\nDecision Boundary: Gini vs Entropy',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('結論：Gini 和 Entropy 通常產生非常相似的樹（約 98% 的情況結果相同）')
print('Gini 計算更快（不需要 log 運算），是 sklearn 的預設值')

---
## Part 4: Random Forest — 隨機森林

隨機森林 = **Bagging** + **特徵隨機子集 (Random Feature Subset)**

兩層隨機性 (Double Randomness)：
1. **資料隨機**：Bootstrap 取樣（有放回抽樣），每棵樹約使用 63.2% 的資料
2. **特徵隨機**：每次分裂只考慮隨機選取的 $m = \sqrt{p}$ 個特徵

**OOB (Out-of-Bag) 估計**：每棵樹約有 36.8% 的資料未被取樣到，
可用來估計泛化誤差，無需額外的驗證集。

$$P(\text{not selected}) = \left(1 - \frac{1}{N}\right)^N \approx \frac{1}{e} \approx 0.368$$

In [ ]:
# Part 4: Random Forest — 不同 n_estimators 的 OOB 分數曲線
# 使用完整的 Iris 資料集（4 個特徵）
X_full = iris.data
y_full = iris.target

n_estimators_range = list(range(1, 201, 5))
oob_scores = []
train_scores = []

for n_est in n_estimators_range:
    rf = RandomForestClassifier(n_estimators=n_est, oob_score=True, random_state=42)
    rf.fit(X_full, y_full)
    oob_scores.append(rf.oob_score_)
    train_scores.append(rf.score(X_full, y_full))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(n_estimators_range, oob_scores, 'o-', color='#2563eb', markersize=3,
        linewidth=1.5, label='OOB Score')
ax.plot(n_estimators_range, train_scores, 's-', color='#dc2626', markersize=3,
        linewidth=1.5, label='Training Score', alpha=0.7)
ax.set_xlabel('Number of Trees (n_estimators)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('隨機森林：n_estimators vs OOB Score\nRandom Forest: OOB Score Curve',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.85, 1.02])

# 標記最佳 OOB 分數
best_idx = np.argmax(oob_scores)
ax.annotate(f'Best OOB={oob_scores[best_idx]:.3f}\nn_est={n_estimators_range[best_idx]}',
            xy=(n_estimators_range[best_idx], oob_scores[best_idx]),
            xytext=(n_estimators_range[best_idx]+30, oob_scores[best_idx]-0.03),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='black'),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print(f'最佳 OOB Score: {max(oob_scores):.4f} (n_estimators={n_estimators_range[best_idx]})')
print('觀察：OOB 分數隨樹的數量增加而穩定，約 50 棵樹後趨於收斂')
print('訓練分數始終接近 1.0，因為隨機森林使用完全生長的深樹（低偏差）')

In [ ]:
# 用特定的 n_estimators 值展示 RF 決策邊界的平滑化
n_est_values = [1, 10, 50, 100]
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, n_est in zip(axes, n_est_values):
    rf = RandomForestClassifier(n_estimators=n_est, random_state=42)
    rf.fit(X_iris, y_iris)
    acc = rf.score(X_iris, y_iris)

    plot_decision_boundary(rf, X_iris, y_iris, ax=ax,
                           title=f'RF n_estimators={n_est}\nAcc={acc:.3f}',
                           class_names=list(class_names))
    ax.set_xlabel('Petal Length')
    ax.set_ylabel('Petal Width')

fig.suptitle('隨機森林：樹數量增加時決策邊界的平滑化\nRandom Forest: Boundary Smoothing with More Trees',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print('觀察：隨著樹的數量增加，決策邊界變得更平滑、穩定')
print('n_estimators=1 時等同於單棵隨機樹，邊界不規則')

---
## Part 5: GBDT — 梯度提升決策樹

GBDT 的核心思想：每一棵新的樹學習的是前面所有樹的**殘差 (Residuals)**。

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

其中 $\eta$ 是學習率 (Learning Rate / Shrinkage)，$h_m(x)$ 是第 $m$ 棵樹。

**學習率的角色：**
- 較小的學習率 → 每棵樹貢獻較小 → 需要更多樹 → 泛化能力通常更好
- 較大的學習率 → 學習更快 → 但容易過擬合
- 經驗法則：小學習率 + 多棵樹 = 更好的泛化效果

In [ ]:
# Part 5: GBDT 學習曲線 — learning_rate 與 n_estimators 的交互影響
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.3, random_state=42)

learning_rates = [0.01, 0.1, 0.3, 1.0]
n_est_range_gb = list(range(1, 201, 5))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_lr = ['#2563eb', '#059669', '#d97706', '#dc2626']

for lr, color in zip(learning_rates, colors_lr):
    train_scores_gb = []
    test_scores_gb = []

    for n_est in n_est_range_gb:
        gb = GradientBoostingClassifier(n_estimators=n_est, learning_rate=lr,
                                        max_depth=3, random_state=42)
        gb.fit(X_train_full, y_train_full)
        train_scores_gb.append(gb.score(X_train_full, y_train_full))
        test_scores_gb.append(gb.score(X_test_full, y_test_full))

    axes[0].plot(n_est_range_gb, train_scores_gb, '-', color=color, linewidth=1.5,
                label=f'lr={lr}')
    axes[1].plot(n_est_range_gb, test_scores_gb, '-', color=color, linewidth=1.5,
                label=f'lr={lr}')

axes[0].set_title('訓練集準確率\nTraining Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('n_estimators')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.5, 1.05])

axes[1].set_title('測試集準確率\nTest Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('n_estimators')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0.5, 1.05])

fig.suptitle('GBDT 學習曲線：learning_rate 的影響\nGBDT Learning Curves: Effect of Learning Rate',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print('觀察重點：')
print('- lr=1.0: 訓練快速達到 1.0，但測試集可能不穩定（過擬合風險高）')
print('- lr=0.01: 收斂慢但穩定，需要更多的樹')
print('- lr=0.1: 通常是不錯的初始選擇')
print('- 經驗法則：使用較小的學習率搭配較多棵樹 → 更好的泛化效果')

In [ ]:
# GBDT staged predictions — 觀察逐步添加樹的效果
gb_staged = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                        max_depth=3, random_state=42)
gb_staged.fit(X_train_full, y_train_full)

train_staged = list(gb_staged.staged_score(X_train_full, y_train_full))
test_staged = list(gb_staged.staged_score(X_test_full, y_test_full))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, 201), train_staged, '-', color='#dc2626', linewidth=2, label='Train')
ax.plot(range(1, 201), test_staged, '-', color='#2563eb', linewidth=2, label='Test')
ax.fill_between(range(1, 201), train_staged, test_staged, alpha=0.1, color='gray')

best_test_iter = np.argmax(test_staged) + 1
ax.axvline(x=best_test_iter, color='green', linestyle='--', alpha=0.7,
           label=f'Best test iter={best_test_iter}')

ax.set_xlabel('Number of Boosting Iterations', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('GBDT Staged Predictions: Train vs Test\nGBDT 逐步預測：觀察過擬合點',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'最佳測試迭代次數: {best_test_iter}')
print(f'最佳測試準確率: {max(test_staged):.4f}')

---
## Part 6: Ensemble Comparison — DT vs RF vs GBDT

讓我們在同一個非線性資料集上全面比較三種模型：

| 方法 | 策略 | 目標 |
|------|------|------|
| Decision Tree | 單一樹 | 基線模型 |
| Random Forest | Bagging + 特徵隨機 | 降低變異 (Reduce Variance) |
| GBDT | Boosting + 殘差學習 | 降低偏差 (Reduce Bias) |

In [ ]:
# Part 6: DT vs RF vs GBDT — 決策邊界與準確率比較
np.random.seed(42)
X_moon, y_moon = make_moons(n_samples=300, noise=0.25, random_state=42)
X_moon_train, X_moon_test, y_moon_train, y_moon_test = train_test_split(
    X_moon, y_moon, test_size=0.3, random_state=42)

models = [
    ('Decision Tree', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)),
    ('GBDT', GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                         learning_rate=0.1, random_state=42)),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
cmap_light_2 = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_bold_2 = ['#FF0000', '#0000FF']

for ax, (name, clf) in zip(axes, models):
    clf.fit(X_moon_train, y_moon_train)
    train_acc = clf.score(X_moon_train, y_moon_train)
    test_acc = clf.score(X_moon_test, y_moon_test)

    # 繪製決策邊界
    x_min, x_max = X_moon[:, 0].min() - 0.5, X_moon[:, 0].max() + 0.5
    y_min, y_max = X_moon[:, 1].min() - 0.5, X_moon[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, cmap=cmap_light_2, alpha=0.4)
    ax.contour(xx, yy, Z, colors='k', linewidths=0.5, alpha=0.5)

    for idx, (color, marker) in enumerate(zip(cmap_bold_2, ['o', 's'])):
        mask_tr = y_moon_train == idx
        mask_te = y_moon_test == idx
        ax.scatter(X_moon_train[mask_tr, 0], X_moon_train[mask_tr, 1],
                   c=color, marker=marker, edgecolors='k', s=30, alpha=0.4)
        ax.scatter(X_moon_test[mask_te, 0], X_moon_test[mask_te, 1],
                   c=color, marker=marker, edgecolors='k', s=60, alpha=0.9,
                   label=f'Test Class {idx}', zorder=3)

    ax.set_title(f'{name}\nTrain={train_acc:.3f}, Test={test_acc:.3f}',
                fontsize=12, fontweight='bold')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.legend(loc='best', fontsize=8)

fig.suptitle('DT vs RF vs GBDT 決策邊界比較 (Moon Dataset)\nEnsemble Comparison',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# 交叉驗證比較
print('\n5-Fold 交叉驗證比較 (5-Fold CV Comparison):')
cv_results = {}
for name, _ in models:
    if name == 'Decision Tree':
        clf_cv = DecisionTreeClassifier(max_depth=5, random_state=42)
    elif name == 'Random Forest':
        clf_cv = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    else:
        clf_cv = GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                            learning_rate=0.1, random_state=42)
    scores = cross_val_score(clf_cv, X_moon, y_moon, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f'  {name:20s}: {scores.mean():.4f} (+/- {scores.std():.4f})')

In [ ]:
# 箱型圖比較 CV 分數分布
fig, ax = plt.subplots(figsize=(8, 5))
box_data = [cv_results[name] for name, _ in models]
box_labels = [name for name, _ in models]
bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True,
                boxprops=dict(linewidth=1.5),
                medianprops=dict(color='black', linewidth=2))

box_colors = ['#3b82f6', '#10b981', '#f59e0b']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_ylabel('CV Accuracy', fontsize=12)
ax.set_title('5-Fold CV 分數比較\n5-Fold CV Score Comparison', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## Part 7: Feature Importance — 特徵重要度比較

樹模型的一大優勢是**內建特徵重要度 (Built-in Feature Importance)**。

在 sklearn 中，`feature_importances_` 基於**加權不純度下降量 (Mean Decrease Impurity, MDI)**：

$$\text{Feature Importance}_j = \sum_{\text{nodes using } j} \frac{N_t}{N} \cdot \Delta \text{Impurity}$$

讓我們比較 DT、RF、GBDT 三種模型的特徵重要度排序。

In [ ]:
# Part 7: 特徵重要度比較 (Wine 資料集 — 13 個特徵)
wine = load_wine()
X_wine = wine.data
y_wine = wine.target
wine_features = wine.feature_names

# 訓練三個模型
dt_wine = DecisionTreeClassifier(max_depth=5, random_state=42)
rf_wine = RandomForestClassifier(n_estimators=200, random_state=42)
gb_wine = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                      max_depth=3, random_state=42)

dt_wine.fit(X_wine, y_wine)
rf_wine.fit(X_wine, y_wine)
gb_wine.fit(X_wine, y_wine)

# 繪製特徵重要度 — 水平長條圖
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
model_names_fi = ['Decision Tree', 'Random Forest', 'GBDT']
model_list_fi = [dt_wine, rf_wine, gb_wine]
colors_fi = ['#3b82f6', '#10b981', '#f59e0b']

for ax, model, mname, color in zip(axes, model_list_fi, model_names_fi, colors_fi):
    importances = model.feature_importances_
    sorted_idx = np.argsort(importances)

    ax.barh(range(len(sorted_idx)), importances[sorted_idx], color=color,
            edgecolor='black', alpha=0.8)
    ax.set_yticks(range(len(sorted_idx)))
    ax.set_yticklabels([wine_features[i] for i in sorted_idx], fontsize=9)
    ax.set_xlabel('Feature Importance (MDI)', fontsize=11)
    ax.set_title(f'{mname}\nAcc={model.score(X_wine, y_wine):.3f}',
                fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

fig.suptitle('特徵重要度比較：DT vs RF vs GBDT (Wine Dataset)\n'
             'Feature Importance Comparison',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('觀察重點：')
print('- Decision Tree: 重要度可能較集中在少數特徵（因為只有一棵樹）')
print('- Random Forest: 由於特徵隨機子集，重要度分配較均勻')
print('- GBDT: 通常能更精確地識別出最關鍵的特徵')
print(f'\nRF Top-3 特徵: ', end='')
rf_top3 = np.argsort(rf_wine.feature_importances_)[-3:][::-1]
print(', '.join([wine_features[i] for i in rf_top3]))

---
## Part 8: Overfitting Demo — 過擬合實驗

未經限制的決策樹會持續生長直到每個葉節點都完全純淨，這通常導致嚴重的過擬合。

讓我們觀察**訓練集 vs 測試集準確率**隨樹深度增加的變化，以及集成方法如何緩解過擬合。

In [ ]:
# Part 8: 過擬合實驗 — 訓練/測試準確率 vs 樹深度
np.random.seed(42)
X_of, y_of = make_moons(n_samples=200, noise=0.3, random_state=42)
X_of_train, X_of_test, y_of_train, y_of_test = train_test_split(
    X_of, y_of, test_size=0.3, random_state=42)

depths_range = range(1, 21)
results_of = {'Decision Tree': {'train': [], 'test': []},
              'Random Forest': {'train': [], 'test': []},
              'GBDT': {'train': [], 'test': []}}

for d in depths_range:
    # Decision Tree
    dt_tmp = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt_tmp.fit(X_of_train, y_of_train)
    results_of['Decision Tree']['train'].append(dt_tmp.score(X_of_train, y_of_train))
    results_of['Decision Tree']['test'].append(dt_tmp.score(X_of_test, y_of_test))

    # Random Forest
    rf_tmp = RandomForestClassifier(n_estimators=50, max_depth=d, random_state=42)
    rf_tmp.fit(X_of_train, y_of_train)
    results_of['Random Forest']['train'].append(rf_tmp.score(X_of_train, y_of_train))
    results_of['Random Forest']['test'].append(rf_tmp.score(X_of_test, y_of_test))

    # GBDT
    gb_tmp = GradientBoostingClassifier(n_estimators=50, max_depth=d,
                                        learning_rate=0.1, random_state=42)
    gb_tmp.fit(X_of_train, y_of_train)
    results_of['GBDT']['train'].append(gb_tmp.score(X_of_train, y_of_train))
    results_of['GBDT']['test'].append(gb_tmp.score(X_of_test, y_of_test))

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
colors_of = {'Decision Tree': '#3b82f6', 'Random Forest': '#10b981', 'GBDT': '#f59e0b'}

for ax, (name, data) in zip(axes, results_of.items()):
    color = colors_of[name]
    ax.plot(list(depths_range), data['train'], 'o-', color=color, linewidth=2,
            markersize=4, label='Train', alpha=0.9)
    ax.plot(list(depths_range), data['test'], 's--', color=color, linewidth=2,
            markersize=4, label='Test', alpha=0.6)
    ax.fill_between(list(depths_range), data['train'], data['test'],
                    alpha=0.1, color=color)
    ax.set_xlabel('max_depth', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0.6, 1.05])

    # 標記最佳測試深度
    best_d = list(depths_range)[np.argmax(data['test'])]
    best_test = max(data['test'])
    ax.axvline(x=best_d, color='red', linestyle=':', alpha=0.5)
    ax.text(best_d + 0.5, 0.65, f'Best depth={best_d}\nTest={best_test:.3f}',
            fontsize=9, color='red')

fig.suptitle('過擬合實驗：Train vs Test Accuracy as Depth Increases\n'
             '訓練 vs 測試準確率隨深度增加的變化',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print('關鍵觀察：')
print('- Decision Tree: Train/Test 差距隨深度增加而擴大 -> 過擬合明顯')
print('- Random Forest: 集成降低了變異，過擬合程度較輕')
print('- GBDT: 使用淺樹 + Boosting，gap 也較可控')

In [ ]:
# 視覺化不同深度 DT 的過擬合效果（決策邊界）
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
selected_depths = [1, 3, 5, 8, 12, 20]
cmap_2 = ListedColormap(['#FFAAAA', '#AAAAFF'])

for ax, d in zip(axes.ravel(), selected_depths):
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_of_train, y_of_train)
    train_a = clf.score(X_of_train, y_of_train)
    test_a = clf.score(X_of_test, y_of_test)

    x_min, x_max = X_of[:, 0].min() - 0.5, X_of[:, 0].max() + 0.5
    y_min, y_max = X_of[:, 1].min() - 0.5, X_of[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, cmap=cmap_2, alpha=0.3)
    ax.contour(xx, yy, Z, colors='k', linewidths=0.3, alpha=0.5)

    ax.scatter(X_of_train[y_of_train == 0, 0], X_of_train[y_of_train == 0, 1],
              c='red', marker='o', edgecolors='k', s=30, alpha=0.5, label='Train 0')
    ax.scatter(X_of_train[y_of_train == 1, 0], X_of_train[y_of_train == 1, 1],
              c='blue', marker='o', edgecolors='k', s=30, alpha=0.5, label='Train 1')
    ax.scatter(X_of_test[y_of_test == 0, 0], X_of_test[y_of_test == 0, 1],
              c='red', marker='x', s=60, alpha=0.9, label='Test 0')
    ax.scatter(X_of_test[y_of_test == 1, 0], X_of_test[y_of_test == 1, 1],
              c='blue', marker='x', s=60, alpha=0.9, label='Test 1')

    ax.set_title(f'depth={d}  |  Train={train_a:.3f}, Test={test_a:.3f}',
                fontsize=11, fontweight='bold')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

axes[0, 0].legend(fontsize=7, loc='lower left')
fig.suptitle('決策樹過擬合：深度增加時邊界越來越不規則\n'
             'DT Overfitting: Boundaries Become Irregular with Depth',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('觀察：深度越大，決策邊界越不規則（出現小島和鋸齒），過擬合訓練資料中的雜訊')

---
## Exercises — 練習題

### 練習 1：嘗試不同的 max_features 對 Random Forest 的影響

使用 Wine 資料集，分別用 `max_features = 'sqrt'`, `'log2'`, `0.5`, `1.0` 訓練隨機森林，
比較 5-fold 交叉驗證分數。用箱型圖 (Box Plot) 視覺化結果。

**提示：** `max_features=1.0` 等於使用全部特徵（退化為 Bagging）。

In [ ]:
# 練習 1：不同 max_features 的 Random Forest
# TODO: 使用 Wine 資料集 (已載入為 X_wine, y_wine)
# TODO: 定義 max_features_list = ['sqrt', 'log2', 0.5, 1.0]
# TODO: 對每個 max_features 訓練 RF 並用 cross_val_score 計算 5-fold CV 分數
# TODO: 用 plt.boxplot() 視覺化比較
# TODO: 觀察哪個 max_features 效果最好


### 練習 2：手動實現簡單的投票集成 (Manual Voting Ensemble)

使用 Moon 資料集，訓練 5 棵不同 `random_state` 的 Decision Tree，
手動實現多數投票 (Majority Voting) 進行預測，並比較集成結果與單一樹的準確率。

**提示：** `scipy.stats.mode` 可以快速計算多數投票。

In [ ]:
# 練習 2：手動投票集成
# from scipy.stats import mode
# TODO: 訓練 5 棵 DecisionTreeClassifier（random_state=0,1,2,3,4）
# TODO: 收集每棵樹對測試集 X_moon_test 的預測到一個 (5, n_test) 矩陣
# TODO: 用 scipy.stats.mode 沿 axis=0 進行多數投票
# TODO: 計算集成準確率 vs 各棵樹的平均準確率
# TODO: 印出比較結果


### 練習 3：在不同資料集上比較 RF 和 GBDT

使用 `load_wine()` 資料集，比較 Random Forest 和 GBDT 在不同 `n_estimators` 下的
5-fold 交叉驗證分數。繪製兩個模型的 CV 分數隨 `n_estimators` 增加的曲線。

**提示：** 使用 `n_estimators = [10, 30, 50, 100, 150, 200]`。

In [ ]:
# 練習 3：RF vs GBDT on Wine Dataset
# TODO: 定義 n_est_list = [10, 30, 50, 100, 150, 200]
# TODO: 對每個 n_estimators，分別計算 RF 和 GBDT 的 5-fold CV 均值與標準差
# TODO: 用 plt.errorbar() 繪製兩條曲線（含誤差棒）
# TODO: 添加圖例、標題，觀察兩者的收斂行為差異


---
## Summary — 本週重點回顧

### 關鍵概念 Key Takeaways

1. **決策樹 (Decision Tree)** 透過 Gini/Entropy 選擇最佳分裂，邊界是軸對齊的矩形區域
2. **max_depth** 是控制過擬合最重要的超參數：太淺 → 欠擬合，太深 → 過擬合
3. **Gini vs Entropy** 實務上差異很小（約 98% 相同），Gini 計算更快是 sklearn 的預設
4. **隨機森林 (Random Forest)** = Bagging + 特徵隨機子集 → **降低變異**
   - 增加樹數通常不會過擬合，OOB Score 可免費估計泛化誤差
5. **GBDT** = Boosting + 殘差學習 → **降低偏差**
   - 學習率與樹的數量需搭配調校，小學習率 + 多樹 = 更好的泛化
6. **特徵重要度** 是樹模型的一大優勢，但基於 MDI 的方法可能偏向高基數特徵
7. 集成方法（RF、GBDT）在測試集上的表現通常優於單棵決策樹

### 下週預告 Next Week Preview
**第 8 週：特徵重要度與 SHAP 值 (Feature Importance & SHAP)** — 深入理解模型可解釋性，
學習排列重要度 (Permutation Importance)、Shapley 值的理論基礎，以及如何用視覺化圖表解讀特徵貢獻。